In [4]:
 # ============================================================
# CELL 1 — INSTALL REQUIRED LIBRARIES
# ============================================================

# Run only once

!pip install -q \
google-cloud-bigquery \
sentence-transformers \
faiss-cpu \
google-generativeai 

In [13]:
!pip install vertexai

In [9]:
# ============================================================
# CELL 2 — IMPORT LIBRARIES
# ============================================================
from google.cloud import bigquery
import vertexai
from vertexai.language_models import TextEmbeddingModel
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd

import google.genai as genai


In [5]:
# ============================================================
# CELL 3 — CREATE BIGQUERY CLIENT
# ============================================================

# Vertex AI Workbench automatically uses
# your logged-in GCP credentials

client = bigquery.Client()

print("BigQuery client created successfully")


BigQuery client created successfully


In [6]:
# ============================================================
# CELL 4 — READ CHUNKED DATA FROM BIGQUERY
# ============================================================

query = """
SELECT
    chunk_id,
    chunk_text,
    topic,
    country,
    chunk_sequence
FROM
`project-2a49d3eb-fbd7-4884-b79.rag_dataset.sdg_chunks`
ORDER BY chunk_sequence
"""

df = client.query(query).to_dataframe()

print("Data loaded from BigQuery")

print(df.head())


Data loaded from BigQuery
   chunk_id                                         chunk_text  \
0         0  introduction as we mark a decade since the ado...   
1         1  compelling case for why the transformative vis...   
2         2  to transform our world by 2030. what we have l...   
3         3  what is possible when the international commun...   
4         4  completion rates rising at all levels, and the...   

                     topic country  chunk_sequence  
0  Sustainable Development  Global               1  
1  Sustainable Development  Global               2  
2  Sustainable Development  Global               3  
3  Sustainable Development  Global               4  
4  Sustainable Development  Global               5  


In [7]:
# ============================================================
# CELL 5 — CHECK DATA
# ============================================================

print("Number of chunks:", len(df))

print("\nColumns:")
print(df.columns)

print("\nExample chunk:\n")
print(df.iloc[0]["chunk_text"])

Number of chunks: 54

Columns:
Index(['chunk_id', 'chunk_text', 'topic', 'country', 'chunk_sequence'], dtype='object')

Example chunk:

introduction as we mark a decade since the adoption of the 2030 agenda for sustainable development, we find ourselves at an inflection point in human history. with five years remaining to achieve the sustainable development goals (sdgs), this report presents both a frank assessment of our current position and a


In [15]:

# ============================================================
# CELL 6 — INITIALIZE VERTEX AI  AND LOAD EMBEDDING MODEL
# ============================================================
import vertexai
from vertexai.language_models import TextEmbeddingModel

PROJECT_ID="project-2a49d3eb-fbd7-4884-b79"
REGION="us-central1"
vertexai.init( project=PROJECT_ID, location=REGION ) 
print("Vertex AI initialized")


embedding_model = TextEmbeddingModel.from_pretrained(
    "text-embedding-004"
)

print("Vertex AI embedding model loaded")



Vertex AI initialized
Vertex AI embedding model loaded


In [16]:
# ============================================================
# CELL 7 — GENERATE EMBEDDINGS
# ============================================================

texts = df["chunk_text"].tolist()

all_embeddings = []

# Vertex AI embedding API works best in batches

batch_size = 5

for i in range(0, len(texts), batch_size):

    batch = texts[i:i+batch_size]

    response = embedding_model.get_embeddings(batch)

    for embedding in response:

        all_embeddings.append(
            embedding.values
        )

print("Embeddings generated")

embeddings = np.array(all_embeddings)

print("Embedding shape:", embeddings.shape)

ResourceExhausted: 429 Quota exceeded for aiplatform.googleapis.com/online_prediction_requests_per_base_model with base model: textembedding-gecko. Please submit a quota increase request. https://cloud.google.com/vertex-ai/docs/generative-ai/quotas-genai.

In [17]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


In [18]:
# ============================================================
# CELL 7 — GENERATE EMBEDDINGS
# ============================================================

texts = df["chunk_text"].tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Embeddings generated")

print("Embedding shape:", embeddings.shape)



Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings generated
Embedding shape: (54, 384)


In [20]:

# ============================================================
# CELL 8 — CREATE FAISS VECTOR INDEX
# ============================================================

dimension = embeddings.shape[1]

# L2 similarity index

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings)
)


print("FAISS index created")

print("Total vectors in index:", index.ntotal)


FAISS index created
Total vectors in index: 54


In [21]:


# ============================================================
# CELL 9 — OPTIONAL SAVE FAISS INDEX
# ============================================================

faiss.write_index(
    index,
    "sdg_vector.index"
)

print("FAISS index saved locally")

FAISS index saved locally


In [22]:
# ============================================================
# CELL 10 — RETRIEVAL FUNCTION
# ============================================================

def retrieve(query, top_k=3):

    # Convert query into embedding vector

    query_embedding = embedding_model.encode([query])  

    # Search FAISS index

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    # Collect retrieved chunks

    retrieved_chunks = []

    for idx in indices[0]:

        retrieved_chunks.append({
            "chunk_id": df.iloc[idx]["chunk_id"],
            "topic": df.iloc[idx]["topic"],
            "country": df.iloc[idx]["country"],
            "text": df.iloc[idx]["chunk_text"]
        })

    return retrieved_chunks

In [30]:
# ============================================================
# CELL 11 — TEST RETRIEVAL
# ============================================================
question= "How can renewable energy help sustainability?"
results = retrieve(
    question
)

print("question")
for i, result in enumerate(results):

    print(f"\nRESULT {i+1}")
    print("=" * 50)

    print("Topic:", result["topic"])
    print("Country:", result["country"])

    print("\nTEXT:\n")
    print(result["text"])

question

RESULT 1
Topic: Sustainable Development
Country: Global

TEXT:

empowered to build better and more resilient futures. they validate the fundamental premise of the 2030 agenda: that sustainable development is achievable when we combine evidence-based policies with sustained political commitment and investment. confronting hard truths however, this report also compels us to confront uncomfortable truths about the challenges that

RESULT 2
Topic: Sustainable Development
Country: Global

TEXT:

introduction as we mark a decade since the adoption of the 2030 agenda for sustainable development, we find ourselves at an inflection point in human history. with five years remaining to achieve the sustainable development goals (sdgs), this report presents both a frank assessment of our current position and a

RESULT 3
Topic: Sustainable Development
Country: Global

TEXT:

no country, regardless of its wealth or capacity, can address climate change, pandemic preparedness or inequality al

In [ ]:
# ============================================================
# CELL 12 — CONFIGURE GEMINI API
# ============================================================

# Replace with your Gemini API key

GOOGLE_API_KEY = ""

client = genai.Client(
    api_key=GOOGLE_API_KEY
)

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain sustainable development goals"
)



print("Gemini configured successfully")

# ============================================================
# CELL 13 — FINAL RAG FUNCTION
# ============================================================

def ask_rag(question):

    # Step 1 — Retrieve relevant chunks

    retrieved_chunks = retrieve(question)

    # Step 2 — Build context text

    context_text = "\n\n".join(
        [chunk["text"] for chunk in retrieved_chunks]
    )

    # Step 3 — Create RAG prompt

    prompt = f"""
    You are a sustainability expert assistant.

    Answer the question using ONLY the context below.

    Context:
    {context_text}

    Question:
    {question}
    """

    # Step 4 — Generate response

    response = gemini_model.generate_content(
        prompt
    )

    return response.text
# ============================================================
# CELL 14 — TEST COMPLETE RAG PIPELINE
# ============================================================

question = """
What are the major sustainable development goals?
"""

response = ask_rag(question)

print("\nQUESTION:")
print(question)

print("\nANSWER:")
print(response)

